In [ ]:
import numpy as np
import pymc as pm
import arviz as az
# -- use this line at the beginning of your notebook to turn on interactive plots
%matplotlib notebook
import scipy.io
# note these imports, thay may be useful in your own task

import matplotlib.pyplot as plt  # plotting library
import numpy as np  # work with numeric arrays without labeled axes
import xarray as xr  # work with arrays with labeled axes
import pandas as pd

prepare the data, grouping the divertor and physical parameters. too many parameters will take to much instanility for the regressing.
Then do the OLS regressing as a reference.

In [ ]:
file_path = 'LH_metal_DB_v1.12.csv'  # Replace with your file path
df = pd.read_csv(file_path)
df.loc[:, 'TOK'] = df['TOK'].str.strip()
df['IP'] = df['IP'].abs() / 1000000
df['BT'] = df['BT'].abs()
df['PLTH'] = df['PLTH'] / 1000000
df['NEL'] = df['NEL'] / 1e19

df.loc[:, 'TOK'] = df['TOK'].str.strip()

reference_dict = {
    1.00:  "Hydrogen",
    2.00:  "Deuterium",
    2.50:  "Deuterium and tritium",
    3.00:  "3He or tritium",
    4.00:  "4He",
    14.01: "Nitrogen"
}

# Define a function to return the corresponding category name based on an observation value PGASA
def categorize_pgasa(value, ref_dict, tolerance=0.2):
    """
    value: The numerical value to be categorized
    ref_dict: {Reference Value: Category Name}
    tolerance: Tolerance level; |value - reference| must be <= this value to be considered the same category
    """
    # 1) Find the reference value closest to the given value
    closest_ref = min(ref_dict.keys(), key=lambda x: abs(x - value))
    # 2) Determine if the difference is within the tolerance level
    if abs(closest_ref - value) <= tolerance:
        return ref_dict[closest_ref]
    else:
        return np.nan  # Or "Other"

# Apply categorize_pgasa to the df["PGASA"] column to generate a new category column
df["Category"] = df["PGASA"].apply(lambda x: categorize_pgasa(x, reference_dict, tolerance=0.1))

# Check classification results if needed
print(df)

# Assume subset1 is your dataframe
colors = df['SELEC2024'].map({1: 'red', 0: 'blue'})
x = df['NEL']
y = df['PLTH']
bt = df['BT'].abs()
ip = df['IP']

# Categorize BT values
# subset2_2 = df[(ip < 2.6) & (ip >=2.4)&(bt < 3.2) & (bt >=2.8)& (df['Category'] == 'Deuterium')& (df['DIVCON'] == 'VT')]
# df = df.drop(subset2_2.index)
print(df)
import statsmodels.api as sm

# Specify the columns you want to include in the new DataFrame
columns_to_include = ['TOK', 'SPLASMA','BT','IP','RGEO','PLTH','NEL','PGASA',"DIVCON"]
new_df = df[df['SELEC2024'] == 1][columns_to_include].copy()
mu0 = 4 * np.pi * 1e-7

new_df['BP'] = 1000000 * (mu0 * new_df['IP']) / (new_df['SPLASMA'] / (2 * np.pi * new_df['RGEO']))

new_df['PLTH/S/2'] = new_df['PLTH'] / (new_df['SPLASMA'] * 2)
# new_df['PLTH/S/2'] = new_df['PLTH']

# Display the new DataFrame
print("New DataFrame:")
print(new_df)

columns_to_include_for_regre = ['BP','PLTH/S/2','IP','BT','NEL','SPLASMA','PGASA']
unique_divcon = new_df['DIVCON'].unique()

# 2. Add new columns to the original DataFrame
for divcon_type in unique_divcon:
    new_df[divcon_type] = (new_df['DIVCON'] == divcon_type).astype(int)
    
multi_df = new_df[columns_to_include_for_regre].copy()
multi_df = multi_df.apply(np.log)

# x = multi_df[['BP','BT','IP','NEL','PGASA']] 
x = multi_df[['BT','IP','NEL','PGASA']]
x[unique_divcon] = new_df[unique_divcon]

# x['Xp'] = x[['Xptref','Xpthigh','Xptin']].max(axis=1)
# x.drop(['STANDARD'], axis=1, inplace=True)
x.drop(['Xptref','Xpthigh','Xptin'], axis=1, inplace=True)
x['CV'] = x[['CORNER','VT']].max(axis=1)
x.drop(['V5E','V5C','V5L','HT3R','STANDARD','CORNER','V5','VT',' VERTPLT'], axis=1, inplace=True)

X_with_intercept = sm.add_constant(x)
y = multi_df["PLTH/S/2"]
model = sm.OLS(y, X_with_intercept)
results = model.fit()

# Print detailed statistical results
print(results.summary())

# Output coefficients and standard errors
coefficients = results.params
std_errors = results.bse

labels = X_with_intercept.columns  # Get variable names
for label, coef, std_err in zip(labels, coefficients, std_errors):
    print(f"{label}: {coef:.3f} ± {std_err:.3f}")

# Calculate and print RMSE
y_pred = results.predict(X_with_intercept)  # Predicted values
rmse = np.sqrt(np.mean((y - y_pred) ** 2))  # Calculate RMSE

print(f"RMSE: {rmse:.3f}")

# Obtain 90% confidence interval
conf_int_90 = results.conf_int(alpha=0.1)

print("\n90% Confidence Interval:\n", conf_int_90)
new_df[columns_to_include_for_regre]

In [ ]:
y=pd.DataFrame(y)
y_pred = pd.DataFrame(y_pred)
y_pred.columns = ['PLTH/S/2']
y['DIVCON']=new_df['DIVCON']
y['TOK']=new_df['TOK']
y['SPLASMA']=new_df['SPLASMA']

y_pred['DIVCON']=new_df['DIVCON']
y_pred['TOK']=new_df['TOK']
y_pred['SPLASMA']=new_df['SPLASMA']
unique_divcon = df['DIVCON'].unique()
y_pred

plot the points with different divertor

main function for the gigh branch. for "Bernoulli" layer, I add it for make the sampling method tobe more stable. it wouldn't effect the result.If you can be sure that the sampling is stable enough(not too many varibles), you can try to delete these layer to be similar with the original paper's structure. In the untitled1.iqynb file, there are a lot of different hyperparemeters setting "try test". they are similar with the lower cell, just has different hyperparameters setting. When I make sure the right parameter setting, I also did a lot of try for different parameters grouping, for example I will set BP & BT & IP be the indepent parameters together or just BP & IP together, and compare the result. In these case, I will change the grouping in the last cell--preparing cell. then use the below cell todo the regressing. 

the last cell is the main result for the high beanch. you can find it in the untitled1.iqynb, after a head cell "call commit".


In [ ]:
# Step 1: Generate simulated data
np.random.seed(42)
N = 3000  # Number of samples
D = 7     # Number of predictors

# Step 2: Define the PyMC model
with pm.Model() as model:
    # Priors
    # Beta for regression coefficients, grouped into classes
    alpha = pm.HalfNormal("alpha", sigma=3.0)  # Dirichlet prior concentration parameter
    # alpha = 1
    p = pm.Dirichlet("p", a=pm.math.ones(20) * alpha, shape=20)  # Class probabilities
    
    # Class assignment for predictors (0-19)
    S = pm.Categorical("S", p=p, shape=D) 
    
    eta = pm.InverseGamma("eta", alpha=4, beta=0.5)  # Variance for beta_m
    mu_0 = pm.Normal("beta_ori", mu=0, sigma=1)
    
    # Regression coefficients for each class
    beta_m = pm.Normal("beta_m", mu=mu_0, sigma=pm.math.sqrt(eta), shape=20)  
    
    # Ensure S only indexes valid categories of beta_m
    beta = pm.Deterministic("beta", beta_m[S])  # Map beta based on S
    
    # Sparsity indicator gamma
    pi = pm.Beta("pi", alpha=1, beta=1)  # Probability of gamma=1
    
    # Initial values for gamma
    gamma_prev = pm.Bernoulli("gamma_prev", p=0.5, shape=D)  
    
    # Control the stability of gamma (the closer to 1, the more stable)
    rho = 0.8  
    gamma = pm.Bernoulli("gamma", p=rho * gamma_prev + (1 - rho) * pi, shape=D)

    # gamma = pm.Bernoulli("gamma", p=pi, shape=D)

    # Noise variance (sigma^2)
    sigma2 = pm.InverseGamma("sigma2", alpha=4, beta=0.1)  # Prior for noise variance

    # Observed data likelihood
    mu = pm.math.dot(X_with_intercept, beta * gamma)  # Predicted mean
    y_obs = pm.Normal("y_obs", mu=mu, sigma=pm.math.sqrt(sigma2), observed=y)

    # Inference
    trace = pm.sample(N, tune=500, chains=4, step=pm.NUTS(target_accept=0.8), return_inferencedata=True)

# Step 3: Analyze and visualize the results
az.plot_trace(trace, var_names=["beta", "gamma", "gamma_prev", "sigma2", "eta", "beta_ori", "pi", "p"])
plt.show()

summary = az.summary(trace, var_names=["beta", "gamma", "gamma_prev", "sigma2", "eta", "beta_ori", "pi", "p"])
print(summary)

In [ ]:
# Randomly draw 1000 samples from the posterior
num_samples = 1000
posterior_indices = np.random.choice(
    trace.posterior["beta"].stack(samples=["chain", "draw"]).shape[1], 
    size=num_samples, 
    replace=False
)
D = 7

# Storage for prediction results
beta_new_look = []
predictions = []
S_new = []
gamma_new = []

for idx in posterior_indices:
    # Extract parameter samples for the specific posterior index
    beta_sample = trace.posterior["beta"].stack(samples=["chain", "draw"]).values[:, idx]
    gamma_sample = trace.posterior["gamma"].stack(samples=["chain", "draw"]).values[:, idx]
    S_sample = trace.posterior["S"].stack(samples=["chain", "draw"]).values[:, idx]
    sigma_sample = np.sqrt(trace.posterior["sigma2"].stack(samples=["chain", "draw"]).values[idx])
    
    # Calculate predicted values
    # Predicted mean
    mu_pred = X_with_intercept @ (beta_sample * gamma_sample)  
    # Sample from the predictive distribution
    y_pred = np.random.normal(mu_pred, sigma_sample, size=X_with_intercept.shape[0])  
    
    S_new.append(gamma_sample * beta_sample)
    beta_new_look.append(S_sample)
    gamma_new.append(gamma_sample)
    predictions.append(y_pred)

S_new1 = np.array(S_new)
gamma_new = np.array(gamma_new)

# Setup plotting grid for the predictors
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(12, 9))
axes = axes.flatten()

for i in range(D):
    ax = axes[i]
    # Filter for cases where gamma equals 1 (Inclusion probability)
    # Currently considering the full distribution as per the original logic
    filtered_beta = S_new1[:, i]
    ax.hist(filtered_beta, bins=30, alpha=0.75, color='blue')
    ax.set_title(f'{X_with_intercept.columns[i]}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('distribution.png', format='png', dpi=300, transparent=True)
plt.show()

# Calculate means and inclusion probabilities
S_mean_new = np.array(S_new).mean(axis=0)
beta_new_look_new = np.array(beta_new_look).mean(axis=0)
gamma_mean = np.array(gamma_new).mean(axis=0) 
gamma_binary = (gamma_mean > 0.5).astype(int)

print("Mean Gamma (Inclusion Probabilities):")
print(gamma_mean)

# Process predictions across the samples
predictions = np.array(predictions).T  
pred_mean = predictions.mean(axis=1) 

print("S Mean New:")
print(S_mean_new)
print("Beta New Look New:")
print(beta_new_look_new)

# Calculate goodness of fit (R-squared)
SS_res = np.sum((pred_mean - y) ** 2)
SS_tot = np.sum((y - np.mean(y)) ** 2)
R_squared = 1 - (SS_res / SS_tot)
print(f"1000 samples' R² values=: {R_squared:.4f}")

In [ ]:
# Save the trace to a NetCDF file for future use
trace.to_netcdf("my_trace_new_for_3_parameter_all.nc")

# Generate and save the statistical summary
# This includes posterior means, standard deviations, and convergence diagnostics (hdi, r_hat, etc.)
summary_df = az.summary(trace, var_names=["beta", "gamma", "gamma_prev", "sigma2", "beta_ori", "eta", "pi", "p"])

# Print the summary to the console
print(summary_df)

# Export the summary to a CSV file for reporting or further analysis
summary_df.to_csv("my_trace_new_for_3_parameter_all.csv")

In [ ]:
import arviz as az
import numpy as np
import matplotlib.pyplot as plt

# Load the MCMC sampling data from the NetCDF file
trace = az.from_netcdf("my_trace_new_for_3_parameter_all.nc")

# Inspect the variables and dimensions stored in the posterior
print(trace.posterior)

just for plot a more clear picture to show the result

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Titles for the predictors
titles = ["$a_0$", "$a_{B_P}$", "$a_{B_T}$", "$a_{I_P}$", "$a_{N_e}$", "$a_{M}$", "D for CORNER/VT"]
S_new1 = np.array(S_new)  # Convert to NumPy array
gamma_new = np.array(gamma_new)

# Plotting the histograms
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(15, 9))
axes = axes.flatten()

for i in range(D):
    ax = axes[i]

    # Apply exp transformation only to the first (i=0) and last (i=D-1) indices
    if i == 0 or i == D - 1:
        filtered_beta = np.exp(S_new1[:, i])  # Apply exp to specific indices
        xlabel_text = 'Value '
    else:
        filtered_beta = S_new1[:, i]
        xlabel_text = 'Value'

    # Calculate the median
    median_value = np.median(filtered_beta)

    # Draw the histogram (color set to light blue)
    ax.hist(filtered_beta, bins=30, alpha=0.75, color='lightblue', edgecolor='black')

    # Mark the median with a dashed red line
    ax.axvline(median_value, color='red', linestyle='dashed', linewidth=2, label=f'Median: {median_value:.2f}')
    ax.legend()
    
    # Set titles and labels
    # ax.set_title(f'{X_with_intercept.columns[i]}')
    ax.set_title(titles[i], fontsize=14)
    ax.set_xlabel(xlabel_text, fontsize=14)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('distribution_exp_partial_median.png', format='png', dpi=300, transparent=True)
plt.show()

# Calculate the posterior means
S_mean_new = S_new1.mean(axis=0)
beta_new_look_new = np.array(beta_new_look).mean(axis=0)  
gamma_mean = np.array(gamma_new).mean(axis=0) 
gamma_binary = (gamma_mean > 0.5).astype(int)

# Print results
print("Gamma Mean (Inclusion Probability):")
print(gamma_mean)
print("S Mean (Coefficients * Gamma):")
print(S_mean_new)
print("Beta New Look Mean (Group Assignment Mean):")
print(beta_new_look_new)

# Calculate R-squared for the predictive mean
predictions = np.array(predictions).T  
pred_mean = predictions.mean(axis=1) 
SS_res = np.sum((pred_mean - y) ** 2)
SS_tot = np.sum((y - np.mean(y)) ** 2)
R_squared = 1 - (SS_res / SS_tot)

print(f"R² value based on 1000 samples: {R_squared:.4f}")

plot the violin picture

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Updated variable name list - Using LaTeX math format
titles = ["$a_0$", "$a_{B_P}$", "$a_{B_T}$", "$a_{I_P}$", 
          "$a_{N_e}$", "$a_{M}$", "$D_{CORNER/VT}$"]

S_new1 = np.array(S_new)  # Convert to NumPy array
gamma_new = np.array(gamma_new)  # Ensure gamma_new is a NumPy array
D = 7  # Total of 7 variables

# Calculate the probability of gamma being 0 for each variable (exclusion probability)
gamma_prob_0 = np.mean(gamma_new < 0.5, axis=0)  # Probability = mean of (gamma < 0.5)

# Prepare data for plotting
data = []
for i in range(D):
    if i == 0 or i == D-1:  # Variables requiring exp transformation
        filtered_beta = np.exp(S_new1[:, i])
    else:
        filtered_beta = S_new1[:, i]
    data.append(filtered_beta)

# Create the figure
plt.figure(figsize=(9, 8))  # Adjust size to fit 7 variables
# Get current axes
ax = plt.gca()

# Find the position for the Divertor term (e.g., CORNER/VT)
# Based on index 6 (the 7th variable)
start_idx = 6  

# Calculate the x-coordinate for the solid horizontal line (violin positions start at 1)
x_start = start_idx + 1 

# Draw a horizontal solid line starting from the Divertor variable
# This often represents a reference or threshold value (e.g., y=1.0)
ax.hlines(y=1.0, xmin=x_start-0.5, xmax=len(data)+0.5, 
          colors='black', linestyles='-', linewidth=1)  

# Create the violin plot
violin_parts = plt.violinplot(data,
                            showmeans=False,
                            showmedians=True,
                            showextrema=True,
                            showextrema=True,
                            widths=0.7)  # Slightly reduce width to fit more variables

# Set colors and styles for the violin bodies
for pc in violin_parts['bodies']:
    pc.set_facecolor('lightblue')
    pc.set_edgecolor('black')
    pc.set_alpha(0.7)

# Style the statistical lines (medians, max, min, bars)
violin_parts['cmedians'].set_color('red')
violin_parts['cmedians'].set_linewidth(2)
violin_parts['cmaxes'].set_color('black')
violin_parts['cmins'].set_color('black')
violin_parts['cbars'].set_color('black')

# Set x-axis labels
xticks_pos = range(1, D+1)
plt.xticks(xticks_pos, titles, rotation=45, ha='center', fontsize=12)

# Add annotations for P(gamma=0) above each violin (only if probability > 0)
for pos, i in zip(xticks_pos, range(D)):
    if gamma_prob_0[i] > 0:  # Only display non-zero probabilities
        prob_text = f"$P$: {gamma_prob_0[i]:.2f}"
        plt.text(pos, plt.ylim()[1]*1.05, prob_text,
                ha='center', va='bottom', fontsize=14, color='darkred')

# Add grid and labels
plt.grid(True, linestyle='--', alpha=0.5)
plt.ylabel('Value', fontsize=12)
plt.title('Posterior Distributions with $P(γ=0)$ Probabilities', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=16)

# Adjust y-axis range to leave room for annotations
ylim = plt.ylim()
plt.ylim(ylim[0], ylim[1]*1.15)  # Increase top space by 15%

# Adjust layout and save/show
plt.tight_layout()
plt.savefig('1violin_plot_partial.png', format='png', dpi=300, transparent=True)
plt.show()

# Calculate remaining statistics
S_mean_new = S_new1.mean(axis=0)
beta_new_look_new = np.array(beta_new_look).mean(axis=0)
gamma_mean = np.array(gamma_new).mean(axis=0)
gamma_binary = (gamma_mean > 0.5).astype(int)

# Print summary results
print("\nGamma means (Inclusion Probabilities):", gamma_mean)
print("S means (Product of Beta and Gamma):", S_mean_new)
print("Beta group assignment means:", beta_new_look_new)

# Calculate predictive R-squared
predictions = np.array(predictions).T  
pred_mean = predictions.mean(axis=1)
SS_res = np.sum((pred_mean - y) ** 2)
SS_tot = np.sum((y - np.mean(y)) ** 2)
R_squared = 1 - (SS_res / SS_tot)

print(f"\n1000 samples' R² values: {R_squared:.4f}")